# Final Project - Scene Reconstruction

## CS445: Computational Photography
### Garima Jajoo (gjajoo2), Jonathan Hautzinger (jmh17), Om Padmani (opadma2), Freddy Jaramillo (fjara3)

In [1]:
import cv2

import numpy as np
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
# %matplotlib widget
%matplotlib tk
import math
from pathlib import Path
from heatmap.height_map import run, select_points
from intelligent_scissors_wrapper import get_inside_mask, get_segments, rgb_save
import os
# %gui tk


# Absolute path to project root
current_dir = Path().resolve()
project_root = current_dir.parent
# datadir = project_root + "src/input_images/"
datadir = "input_images/"
INPUT_IMAGE_DIRECTORY = "input_images"

In [2]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output

def prompt_selection(image,num_clicks,title):
    fig = plt.figure(figsize=(15,10))
    plt.imshow(image, cmap='gray')
    plt.title(title)
    plt.axis('off')

    points = plt.ginput(num_clicks)

    plt.close(fig)
    clear_output(wait=True)

    clicked = np.array(points, dtype=np.float32)
    return clicked

In [6]:
im_file = datadir + 'giza-sideview.png' # must use a side view image
im = np.float32(cv2.imread(im_file, cv2.COLOR_BGR2RGB) / 255.0)

In [7]:
def calculate_heights(vanishing_line, ref_object, ref_height, target_points):
    '''
    Inputs:
        vanishing_line: 2 x 2 numpy array that represents 2 points on the vanishing line
        ref_object: 2x2 numpy array that represents the bottom point and top point of a reference object
        ref_height: int representation of the height of reference object
        target_points: 2n x 2 numpy array that represents the bottom and top points for n objects.
        The bottom point of an object is always followed by the top point.
        
    Output:
        Returns a list of heights. The order is respective to the order of the target points.
    '''  
    num_points=target_points.shape[0]
    assert(num_points%2==0)
    
    # appending the 1 for homogeneous coordinates
    vx=np.array([vanishing_line[0,0], vanishing_line[0,1], 1])
    vy=np.array([vanishing_line[1,0], vanishing_line[1,1], 1])
    b_ref=np.array([ref_object[0,0],ref_object[0,1],1])
    t_ref=np.array([ref_object[1,0],ref_object[1,1],1])
    homogeneous_targets = np.zeros([num_points,3])
    for i in range(num_points):
        homogeneous_targets[i]=np.array([target_points[i,0], target_points[i,1],1])

    height_list=[ref_height]
    for i in range(0,num_points,2):
        b_tar = np.array([homogeneous_targets[i,0],homogeneous_targets[i,1],1])
        t_tar = np.array([homogeneous_targets[i+1,0],homogeneous_targets[i+1,1],1])
        
        # horizon
        horizon = np.cross(vx, vy)
        
        # vanishing point along ground direction between objects
        v = np.cross(np.cross(b_ref, b_tar), horizon)
        v = v / v[2]
        
        t_transfer = np.cross(np.cross(v, t_ref), np.cross(t_tar,b_tar))
        t_transfer = t_transfer / t_transfer[2]
        
        # height ratio
        height = ref_height * abs(b_tar[1] - t_tar[1]) / abs(b_tar[1] - t_transfer[1]) 
        height_list.append(height)

    return height_list
    

In [8]:
vps = prompt_selection(im, 2, 'Click on 2 points on the vanishing line')
ref = prompt_selection(im, 2, 'Click on the bottom point of an object with a known height. Then click on the top of that object.')
targets = prompt_selection(im, -1, 'Click on the bottom and top points of as many objects. When you are done selecting objects, press Enter')
ref_height = 23.2
height_list = calculate_heights(vps,ref,ref_height,targets)

In [9]:
points=[]

points=select_points(datadir + "GizaZoomedSatellite.png") # must use top-down/bird's eye view image
while len(points) != len(height_list):
    print(f"Selected {len(points)} points, but need {len(height_list)}")
    points=select_points(datadir + "GizaZoomedSatellite.png")

In [10]:
print(points)

for index in range(len(height_list)):
    points[index][2] = height_list[index]

print(points)

[[193, 319, None], [325, 488, None]]
[[193, 319, 23.2], [325, 488, 17.005415589053907]]


In [11]:
run(datadir + "GizaZoomedSatellite.png", refs=points)

Loading weights:   0%|          | 0/503 [00:00<?, ?it/s]

In [12]:
input_image = 'GizaZoomedSatellite.png'
input_image_path = datadir + input_image
rgb_save(input_image_path, input_image)

Saved RGB image to: /Users/garimajajoo/Desktop/projects/satelite_image_scene_reconstruction/src/input_images/rgb_foellinger-top-down.png


('/Users/garimajajoo/Desktop/projects/satelite_image_scene_reconstruction/src/input_images/rgb_foellinger-top-down.png',
 'rgb_foellinger-top-down.png')

Run the intelligent_scissors_wrapper.py with the above saved image

./intelligent-scissors-wrapper.py [file-path] --alpha

In [13]:
print(datadir + "rgb_" + input_image)
output_image_path = datadir + "rgb_" + input_image

input_images/rgb_foellinger-top-down.png


In [15]:
!python intelligent_scissors_wrapper.py $output_image_path

Saved segment mask image to: /Users/garimajajoo/Desktop/projects/satelite_image_scene_reconstruction/src/masks/mask_rgb_foellinger-top-down.npy
Figure(640x480)


In [20]:
from Scene_Reconstruction.scene_reconstruction import run_reconstruction_pipeline

outputs = run_reconstruction_pipeline(
    name="giza",
    depth_path="heatmap/output/GizaZoomedSatellite_height_map.npy",
    rgb_path="input_images/rgb_GizaZoomedSatellite.png",
    mask_path="masks/mask_GizaZoomedSatellite.npy",
    output_dir="outputs",
    min_size=30,
    mask_threshold=0.5,
    smooth=True,
    smooth_method="gaussian",
    sigma=1.0,
    sample_step=1,
    make_plots=True,
    show=False
)

FileNotFoundError: [Errno 2] No such file or directory: 'inputs/louvreCroppedSatellite_depth_norm.npy'